# DanmakuRender：Colab 单场录制实验

手动启动，在 Colab 完成录制、ASS 弹幕渲染及 Google Drive 备份，**不投稿、不删除源文件、不自动释放运行时**。

先选择 GPU 运行时。首次使用前，确认自己的 fork 已包含 `codex/colab-poc` 分支及 Colab 入口。

按顺序运行单元格。请勿直接上传整个本地项目或登录目录到公开仓库。首次运行需要本人授权 Drive，并将现有 B 站登录 JSON 放入私有 Drive 的 `DMRColab/credentials/bilibili.json`。Cookie 不写入 Notebook。


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/zaixiakongyiji/DanmakuRenderForColab.git"
BRANCH = "codex/colab-poc"
PROJECT = Path("/content/DanmakuRenderForColab")

def checked(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)

if not PROJECT.exists():
    checked(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(PROJECT)])
else:
    actual_remote = subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip()
    actual_branch = subprocess.check_output(["git", "-C", str(PROJECT), "branch", "--show-current"], text=True).strip()
    if actual_remote != REPOSITORY or actual_branch != BRANCH:
        raise RuntimeError("已有目录不对应选定仓库或分支，请先检查目录。")
    print("使用现有检出；不覆盖运行中的文件。更新代码前请结束当前任务。")
COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("本次源码 commit:", COMMIT)


## 安装依赖

首版只安装 B 站录制及渲染路径所需依赖。系统 FFmpeg 必须实际通过 NVENC + ASS 预检；失败时检查驱动和 FFmpeg 版本，不自动改成 CPU 编码。


In [ ]:
checked(["apt-get", "update", "-qq"])
checked(["apt-get", "install", "-y", "ffmpeg", "fontconfig", "fonts-noto-cjk"])
checked([sys.executable, "-m", "pip", "install", "-r", str(PROJECT / "colab_support/requirements.txt")])
checked(["fc-cache", "-f"])


## 授权 Drive

授权后，本 Notebook 运行的代码可访问挂载的 Drive。观看 Cookie 只用于读取 B 站直播流，备份只包含录像、ASS、成品、运行清单及白名单事件日志。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
COOKIE_FILE = DRIVE_ROOT / "DMRColab/credentials/bilibili.json"
if not COOKIE_FILE.is_file():
    raise FileNotFoundError("请手动上传现有登录 JSON 到 MyDrive/DMRColab/credentials/bilibili.json，再重新运行此单元格。")
print("已找到观看 Cookie 文件，内容不会显示。")


## 运行参数

每次重新录制必须生成新的运行编号。首次建议使用 300 秒分段，并先做两个完整分段的短试跑。`MAX_RECORD_HOURS` 可临时改成 `0.2`（12 分钟），用于验证受控收尾；完整验收还需测试自然下播。


In [ ]:
from datetime import datetime
from uuid import uuid4

ROOM_URL = ""  # 在此填写本次测试直播间的完整链接，勿将个人参数提交到公开仓库。
if not ROOM_URL.strip():
    raise ValueError("请先填写 ROOM_URL，例如 https://live.bilibili.com/<直播间号>。")
SEGMENT_SECONDS = 300
INITIAL_WAIT_MINUTES = 15
OFFLINE_GRACE_MINUTES = 3
MAX_RECORD_HOURS = 12
DRAIN_TIMEOUT_HOURS = 2
MIN_FREE_GIB = 10
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8]
RUN_DIR = Path("/content/dmr-runs") / RUN_ID
BACKUP_DIR = DRIVE_ROOT / "DMRColab/runs" / RUN_ID
COMMAND = [sys.executable, str(PROJECT / "colab_run.py"),
           "--url", ROOM_URL, "--run-dir", str(RUN_DIR),
           "--drive-root", str(DRIVE_ROOT), "--cookie", str(COOKIE_FILE),
           "--segment", str(SEGMENT_SECONDS),
           "--initial-wait", str(INITIAL_WAIT_MINUTES * 60),
           "--offline-grace", str(OFFLINE_GRACE_MINUTES * 60),
           "--max-record", str(MAX_RECORD_HOURS * 3600),
           "--drain-timeout", str(DRAIN_TIMEOUT_HOURS * 3600),
           "--min-free-gib", str(MIN_FREE_GIB)]
print("本地工作目录:", RUN_DIR)
print("Drive 备份目录:", BACKUP_DIR)


In [ ]:
checked(COMMAND + ["--preflight-only"], cwd=PROJECT)
from IPython.display import Video, display
display(Video(str(RUN_DIR / "preflight/rendered.mp4"), embed=True))


## 开始运行

下面的单元格持续等待任务完成，并显示状态、分段数和失败项。首次未开播最多等 15 分钟；下播持续 3 分钟后等待尾段、渲染和备份。

**需要提前停止时，点一次停止单元格**：捕获中断后请求受控停止，仍等待已有分段处理完。再次中断属于强制停止，不算正常完成。遇到浏览器未能传递中断时，可以在运行时终端执行 `touch /content/dmr-runs/<运行编号>/STOP`，然后重新查看结果。

此单元格保持前台执行，不使用后台保活脚本；运行完毕后保留运行时，由你检查并手动释放。


In [ ]:
import json
import time

process = subprocess.Popen(COMMAND, cwd=PROJECT)
last_status = None
stop_requested = False
try:
    while process.poll() is None:
        try:
            manifest_path = RUN_DIR / "manifest.json"
            if manifest_path.exists():
                try:
                    state = json.loads(manifest_path.read_text(encoding="utf-8"))
                    brief = (state.get("status"), len(state.get("segments", {})),
                             tuple(state.get("errors", [])), tuple(state.get("warnings", [])))
                    if brief != last_status:
                        print("状态 / 分段数 / 错误 / 提醒:", brief, flush=True)
                        last_status = brief
                except (OSError, ValueError):
                    pass
            time.sleep(5)
        except KeyboardInterrupt:
            if stop_requested:
                raise
            stop_requested = True
            (RUN_DIR / "STOP").touch()
            print("已请求受控停止，等待最后分段、渲染和备份。", flush=True)
finally:
    if process.poll() is None:
        # 通知监督进程，不直接杀掉只负责启动的父进程留下孤儿任务。
        process.send_signal(2)
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.send_signal(2)
            process.wait(timeout=30)
print("退出码:", process.returncode)
if process.returncode != 0:
    print("本次未完整成功，请查看以下结果和本地 console.log；勿释放运行时中的未备份文件。")


## 检查结果并记录 CU

运行成功需要三个条件：程序退出码为 0、本地清单为 success、Drive 中的清单及文件可读取。校验只证明挂载目录读回一致；还请从 Drive 网页抽查成品。

检查音视频和弹幕同步，核对最后分段。CU 由 Colab 界面手动填写，不从 GPU 型号推算。


In [ ]:
manifest_path = RUN_DIR / "manifest.json"
if not manifest_path.exists():
    raise RuntimeError("尚未生成运行清单，请查看本地 console.log。")
result = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(result, ensure_ascii=False, indent=2))
cloud_manifest = BACKUP_DIR / "manifest.json"
print("Drive 清单可读:", cloud_manifest.is_file())
if cloud_manifest.is_file():
    cloud_result = json.loads(cloud_manifest.read_text(encoding="utf-8"))
    print("Drive 状态:", cloud_result.get("status"))
for key, segment in result.get("segments", {}).items():
    duration = segment.get("duration", 0)
    elapsed = segment.get("render_elapsed_seconds")
    print(key, "弹幕条数:", segment.get("danmaku_dialogues"),
          "渲染耗时/视频时长:", round(elapsed / duration, 3) if duration and elapsed else None)
CU_BEFORE = None  # 填入开始前界面显示的余额
CU_AFTER = None   # 填入结束后界面显示的余额
if CU_BEFORE is not None and CU_AFTER is not None:
    usage = {"cu_before": CU_BEFORE, "cu_after": CU_AFTER, "cu_used": CU_BEFORE - CU_AFTER}
    (RUN_DIR / "usage.json").write_text(json.dumps(usage, indent=2), encoding="utf-8")
    if cloud_manifest.is_file():
        sys.path.insert(0, str(PROJECT))
        from colab_support.core import verified_copy
        verified_copy(RUN_DIR / "usage.json", BACKUP_DIR / "usage.json")


检查完后，手动选择 **运行时 → 断开连接并删除运行时**。先确认需要的录像和诊断文件已经保存；临时磁盘内容不会随 Notebook 保留。
